# Notebook 04: ConvNeXt-Nano Two-Stage ABMIL Training

In [1]:
import os
import json
import time
import numpy as np
import gc
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import (
    f1_score, recall_score, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"timm    : {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

PyTorch : 2.10.0+cu128
CUDA    : True
timm    : 1.0.26
GPU     : Tesla T4


In [2]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
OUT  = Path("/kaggle/working")

X_TRAIN_PATH      = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH      = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH= NB02 / "bag_ids_train.npy"
CLASS_WEIGHTS_PATH= NB02 / "class_weights.json"

In [3]:
MODEL_NAME = "convnext_nano"
PATCH_SIZE = 224
SEED       = 42

BS_STAGE1  = 128
EPOCHS_S1  = 15
PATIENCE_S1= 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [4]:
# import sys
# sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
# from abmil_common import (
#     build_backbone, PatchDataset, PatchClassifier,
#     AttentionPool, BagClassifier, CachedBagDataset, collate_cached,
#     extract_features, compute_all_metrics, get_normalisation_tensors, run_cv, train_stage1_fold,
# )

import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchClassifier, get_normalisation_tensors, train_stage1_fold,IndexedPatchDataset
)

In [5]:
print("Loading arrays...")
X_train_all  = np.load(X_TRAIN_PATH)
y_train_all  = np.load(Y_TRAIN_PATH)
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)

fold_ids = np.load(NB02 / "fold_ids.npy")

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

print(f"X_train_all : {X_train_all.shape}  dtype={X_train_all.dtype}")
print(f"y_train_all : unique={np.unique(y_train_all)}  "
      f"benign={(y_train_all==0).sum()}  "
      f"malignant={(y_train_all==1).sum()}")
print(f"fold_ids : {fold_ids.shape}  unique folds={sorted(set(fold_ids))}")
print(f"Class weights   : {class_weight_dict}")

Loading arrays...
X_train_all : (58820, 224, 224, 1)  dtype=float32
y_train_all : unique=[0 1]  benign=29225  malignant=29595
fold_ids : (1226,)  unique folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Class weights   : {0: 1.0063301967493585, 1: 0.9937489440783916}


In [6]:
STAGE1_PARAMETER_PATH = Path("/kaggle/input/notebooks/mfjmrizvi/2-6-optuna-run-all/convnext_nano_stage1_optuna_study.json")
with open(STAGE1_PARAMETER_PATH) as f:
    parameter = json.load(f)["best_params"]

OPTIMISER_NAME = parameter["optimiser"]
LR             = parameter["lr"]
WEIGHT_DECAY   = parameter["weight_decay"]
print(f"Using parameters: {parameter}")
all_bags = np.unique(bag_ids_all)

Using parameters: {'optimiser': 'Adam', 'lr': 2.0514306954965613e-05, 'weight_decay': 5.292469708762985e-06}


In [7]:
def get_fold_split(fold, all_bags, fold_ids, bag_ids_all, model_name, out_dir):
    if fold == "full":
        rng = np.random.default_rng(999)
        shuffled = rng.permutation(all_bags)
        n_val = int(0.1 * len(shuffled))
        val_bag_indices   = shuffled[:n_val]
        train_bag_indices = shuffled[n_val:]
        save_path = out_dir / f"{model_name}_stage1_full.pth"
    else:
        train_bag_indices = all_bags[fold_ids[all_bags] != fold]
        val_bag_indices   = all_bags[fold_ids[all_bags] == fold]
        save_path = out_dir / f"{model_name}_stage1_fold{fold}.pth"

    train_idx = np.where(np.isin(bag_ids_all, train_bag_indices))[0]
    val_idx   = np.where(np.isin(bag_ids_all, val_bag_indices))[0]

    return train_idx, val_idx, save_path

In [8]:
def evaluate_checkpoint(model_name, save_path, X_all, y_all, val_idx,
                         device, patch_size=224, batch_size=128):
    """Reload a saved Stage 1 checkpoint and compute val_loss/val_auc
    without training. Used to backfill fold_histories after a restart
    skips a fold that's already checkpointed."""
    backbone = build_backbone(model_name, pretrained=False).to(device)
    with torch.no_grad():
        _dummy = torch.zeros(2, 3, patch_size, patch_size, device=device)
        feat_dim = backbone(_dummy).shape[1]
    del _dummy

    patch_model = PatchClassifier(backbone, feat_dim).to(device)
    patch_model.load_state_dict(torch.load(save_path, map_location=device))
    patch_model.eval()

    mean_gpu, std_gpu = get_normalisation_tensors(device)
    val_dl = DataLoader(IndexedPatchDataset(X_all, y_all, val_idx), batch_size=batch_size,
                         shuffle=False, num_workers=2, pin_memory=True)

    pos_weight_val = class_weight_dict[1] / class_weight_dict[0]
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val, device=device))

    val_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for patches, labels in val_dl:
            patches, labels = patches.to(device), labels.to(device)
            patches = patches.repeat(1, 3, 1, 1)
            patches = (patches - mean_gpu) / std_gpu
            with torch.amp.autocast('cuda'):
                logits = patch_model(patches)
                loss_val = criterion(logits, labels)
            val_loss += loss_val.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss /= len(val_dl.dataset)
    val_auc = roc_auc_score(all_labels, all_probs)

    del backbone, patch_model
    gc.collect(); torch.cuda.empty_cache()
    return val_loss, val_auc


In [9]:
fold_histories = {}
for fold in [0, 1, 2, 3, 4, "full"]:
    print(f"\n{'='*60}\nTraining Stage 1: {MODEL_NAME} fold={fold}\n{'='*60}")
    train_idx, val_idx, save_path = get_fold_split(
        fold, all_bags, fold_ids, bag_ids_all, MODEL_NAME, OUT
    )
    print(f"Train patches: {len(train_idx)}  Val patches: {len(val_idx)}")

    if save_path.exists():
        print(f"✓ Checkpoint exists, evaluating only: {save_path.name}")
        best_val_loss, val_auc = evaluate_checkpoint(
            MODEL_NAME, save_path, X_train_all, y_train_all, val_idx, DEVICE
        )
        history = {"note": "reconstructed post-restart, not retrained",
                    "val_loss": best_val_loss, "val_auc": val_auc}
        print(f"    val_loss={best_val_loss:.4f}  auc={val_auc:.4f}")
    else:
        history, best_val_loss = train_stage1_fold(
            model_name=MODEL_NAME,
            X_all=X_train_all, y_all=y_train_all,
            train_idx=train_idx, val_idx=val_idx,
            class_weight_dict=class_weight_dict,
            optimiser_name=OPTIMISER_NAME, lr=LR, weight_decay=WEIGHT_DECAY,
            save_path=save_path, device=DEVICE,
            epochs=EPOCHS_S1, patience=PATIENCE_S1
        )

    fold_histories[str(fold)] = {
        "best_val_loss": best_val_loss,
        "parameters": parameter,
        "epochs_budget": EPOCHS_S1,
        "patience": PATIENCE_S1,
        "train_patches": len(train_idx),
        "val_patches": len(val_idx),
        "history": history,
    }
    gc.collect(); torch.cuda.empty_cache()

    with open(OUT / f"{MODEL_NAME}_stage1_all_folds_summary.json", "w") as f:
        json.dump(fold_histories, f, indent=2)


Training Stage 1: convnext_nano fold=0
Train patches: 47252  Val patches: 11568


model.safetensors:   0%|          | 0.00/62.4M [00:00<?, ?B/s]

  Ep 01/15  train=0.6286  val=0.6351  auc=0.6803
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 02/15  train=0.5738  val=0.6329  auc=0.6962
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 03/15  train=0.5405  val=0.6291  auc=0.7073
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 04/15  train=0.5022  val=0.6659  auc=0.6883
  Ep 05/15  train=0.4492  val=0.7281  auc=0.7057
  Ep 06/15  train=0.3859  val=0.7986  auc=0.6805
  Ep 07/15  train=0.2924  val=0.8968  auc=0.6793
  Ep 08/15  train=0.1547  val=1.1651  auc=0.6701
  Early stopping at epoch 8

Training Stage 1: convnext_nano fold=1
Train patches: 47355  Val patches: 11465
  Ep 01/15  train=0.6266  val=0.6784  auc=0.6814
    ✓ Saved: convnext_nano_stage1_fold1.pth
  Ep 02/15  train=0.5522  val=0.6824  auc=0.6592
  Ep 03/15  train=0.5108  val=0.7136  auc=0.6476
  Ep 04/15  train=0.4597  val=0.7657  auc=0.6333
  Ep 05/15  train=0.3886  val=0.8724  auc=0.6279
  Ep 06/15  train=0.2488  val=0.9962  auc=0.6512
  Early stopping at epoch 6



In [10]:
fold_histories = {}

for fold in [0, 1, 2, 3, 4, "full"]:
    print(f"\n{'='*60}\nTraining Stage 1: {MODEL_NAME} fold={fold}\n{'='*60}")

    train_idx, val_idx, save_path = get_fold_split(
        fold, all_bags, fold_ids, bag_ids_all, MODEL_NAME, OUT
    )
    print(f"Train patches: {len(train_idx)}  Val patches: {len(val_idx)}")

    history, best_val_loss = train_stage1_fold(
        model_name=MODEL_NAME,
        X_all=X_train_all, y_all=y_train_all,
        train_idx=train_idx, val_idx=val_idx,
        class_weight_dict=class_weight_dict,
        optimiser_name=OPTIMISER_NAME, lr=LR, weight_decay=WEIGHT_DECAY,
        save_path=save_path, device=DEVICE,
        epochs=EPOCHS_S1, patience=PATIENCE_S1
    )

    fold_histories[str(fold)] = {
        "best_val_loss": best_val_loss,
        "parameters": parameter,
        "epochs_budget": EPOCHS_S1,
        "patience": PATIENCE_S1,
        "train_patches": len(train_idx),
        "val_patches": len(val_idx),
        "history": history,
    }

    gc.collect(); torch.cuda.empty_cache()


Training Stage 1: convnext_nano fold=0
Train patches: 47252  Val patches: 11568
  Ep 01/15  train=0.6291  val=0.6560  auc=0.6789
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 02/15  train=0.5654  val=0.6469  auc=0.6554
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 03/15  train=0.5261  val=0.6125  auc=0.7267
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 04/15  train=0.4809  val=0.7054  auc=0.6875
  Ep 05/15  train=0.4165  val=0.7500  auc=0.6959
  Ep 06/15  train=0.3311  val=0.9067  auc=0.7012
  Ep 07/15  train=0.2318  val=1.0281  auc=0.6774
  Ep 08/15  train=0.1019  val=1.2535  auc=0.6884
  Early stopping at epoch 8

Training Stage 1: convnext_nano fold=1
Train patches: 47355  Val patches: 11465
  Ep 01/15  train=0.6364  val=0.6588  auc=0.6471
    ✓ Saved: convnext_nano_stage1_fold1.pth
  Ep 02/15  train=0.5522  val=0.7248  auc=0.6376
  Ep 03/15  train=0.5128  val=0.7000  auc=0.6657
  Ep 04/15  train=0.4579  val=0.7273  auc=0.6519
  Ep 05/15  train=0.3931  val=0.8498  auc=0.65

In [11]:
with open(OUT / f"{MODEL_NAME}_stage1_all_folds_summary.json", "w") as f:
    json.dump(fold_histories, f, indent=2)

print("\nAll 6 Stage 1 backbones trained (5 CV folds + 1 production).")


All 6 Stage 1 backbones trained (5 CV folds + 1 production).


In [12]:
# STAGE1_FOLD = 0
# all_bags = np.unique(bag_ids_all)
# train_bag_indices = all_bags[fold_ids[all_bags] != STAGE1_FOLD]
# val_bag_indices   = all_bags[fold_ids[all_bags] == STAGE1_FOLD]

# train_mask = np.isin(bag_ids_all, train_bag_indices)
# val_mask   = np.isin(bag_ids_all, val_bag_indices)

# X_tr, y_tr, bag_tr    = X_train_all[train_mask], y_train_all[train_mask], bag_ids_all[train_mask]
# X_val, y_val, bag_val = X_train_all[val_mask],   y_train_all[val_mask],   bag_ids_all[val_mask]

# print(f"Stage 1 fixed split (fold {STAGE1_FOLD} held out): "
#       f"train_bags={len(train_bag_indices)}  val_bags={len(val_bag_indices)}")

In [13]:
# patch_model = PatchClassifier(backbone, FEAT_DIM).to(DEVICE)

# pos_weight_val = class_weight_dict[1] / class_weight_dict[0]
# criterion_s1   = nn.BCEWithLogitsLoss(
#     pos_weight=torch.tensor(pos_weight_val, device=DEVICE)
# )
# print(f"pos_weight : {pos_weight_val:.4f}")

# optimiser_s1 = optim.Adam(patch_model.parameters(), lr=LR_STAGE1)
# scheduler_s1 = optim.lr_scheduler.ReduceLROnPlateau(
#     optimiser_s1, mode='min', factor=0.5, patience=3
# )

# train_patch_dl = DataLoader(
#     PatchDataset(X_tr, y_tr),
#     batch_size=BS_STAGE1, shuffle=True,
#     num_workers=2, pin_memory=True
# )
# val_patch_dl = DataLoader(
#     PatchDataset(X_val, y_val),
#     batch_size=BS_STAGE1, shuffle=False,
#     num_workers=2, pin_memory=True
# )
# print(f"Train batches : {len(train_patch_dl)}")
# print(f"Val batches   : {len(val_patch_dl)}")

In [14]:
# scaler = torch.amp.GradScaler('cuda')

# s1_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
# best_val_loss_s1    = float("inf")
# patience_counter_s1 = 0
# t0 = time.time()

# for epoch in range(1, EPOCHS_S1 + 1):
#     patch_model.train()
#     running_loss = 0.0
#     for patches, labels in train_patch_dl:
#         # 1. Push raw single-channel patches to GPU memory immediately
#         patches, labels = patches.to(DEVICE), labels.to(DEVICE) 
        
#         # 2. Parallelised channel expansion and normalisation on GPU cores
#         patches = patches.repeat(1, 3, 1, 1)                   
#         patches = (patches - _mean_gpu) / _std_gpu             
        
#         optimiser_s1.zero_grad()
        
#         # 3. Forward pass wrapped in modern, unified AMP autocast
#         with torch.amp.autocast('cuda'):
#             logits = patch_model(patches)
#             loss = criterion_s1(logits, labels)
            
#         scaler.scale(loss).backward()
#         scaler.step(optimiser_s1)
#         scaler.update()
        
#         running_loss += loss.item() * len(labels)
#     train_loss = running_loss / len(train_patch_dl.dataset)

#     # Validation Phase
#     patch_model.eval()
#     val_loss, all_probs, all_labels = 0.0, [], []
#     with torch.no_grad():
#         for patches, labels in val_patch_dl:
#             patches, labels = patches.to(DEVICE), labels.to(DEVICE)
            
#             # Identical fast GPU transformations
#             patches = patches.repeat(1, 3, 1, 1)
#             patches = (patches - _mean_gpu) / _std_gpu
            
#             with torch.amp.autocast('cuda'):
#                 logits   = patch_model(patches)
#                 loss_val = criterion_s1(logits, labels)
                
#             val_loss += loss_val.item() * len(labels)
#             all_probs.extend(torch.sigmoid(logits).cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
            
#     val_loss  /= len(val_patch_dl.dataset)
#     all_probs  = np.array(all_probs)
#     all_labels = np.array(all_labels)
#     val_preds  = (all_probs >= 0.5).astype(int)
#     val_auc    = roc_auc_score(all_labels, all_probs)
#     val_f1     = f1_score(all_labels, val_preds, zero_division=0)

#     s1_history["train_loss"].append(train_loss)
#     s1_history["val_loss"].append(val_loss)
#     s1_history["val_auc"].append(val_auc)
#     s1_history["val_f1"].append(val_f1)
#     scheduler_s1.step(val_loss)

#     print(f"Ep {epoch:02d}/{EPOCHS_S1}  "
#           f"train={train_loss:.4f}  val={val_loss:.4f}  "
#           f"auc={val_auc:.4f}  f1={val_f1:.4f}")

#     if val_loss < best_val_loss_s1:
#         best_val_loss_s1    = val_loss
#         patience_counter_s1 = 0
#         torch.save(patch_model.state_dict(), S1_WEIGHTS)
#         print("  ✓ Saved best Stage 1 weights")
#     else:
#         patience_counter_s1 += 1
#         if patience_counter_s1 >= PATIENCE_S1:
#             print(f"  Early stopping at epoch {epoch}")
#             break

# print(f"\nStage 1 complete: {(time.time()-t0)/60:.1f} min")

In [15]:
# # Stage 1 curves
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(s1_history["train_loss"], label="Train loss")
# axes[0].plot(s1_history["val_loss"],   label="Val loss")
# axes[0].set_title("Stage 1: Loss");     axes[0].legend()
# axes[0].set_xlabel("Epoch")
# axes[1].plot(s1_history["val_auc"], color="orange", label="Val AUC")
# axes[1].set_title("Stage 1: Val AUC"); axes[1].legend()
# axes[1].set_xlabel("Epoch")
# plt.tight_layout()
# plt.savefig(OUT / "convnext_nano_stage1_curves.png", dpi=150)
# plt.show()
# print("Saved convnext_nano_stage1_curves.png")

In [16]:
# patch_model.load_state_dict(
#     torch.load(S1_WEIGHTS, map_location=DEVICE))
# patch_model.eval()

# all_probs, all_labels = [], []
# with torch.no_grad():
#     for patches, labels in val_patch_dl:
#         patches = patches.to(DEVICE)
#         patches = patches.repeat(1, 3, 1, 1)
#         patches = (patches - _mean_gpu) / _std_gpu
        
#         with torch.amp.autocast('cuda'):
#             all_probs.extend(
#                 torch.sigmoid(patch_model(patches)).cpu().numpy())
#         all_labels.extend(labels.numpy())

# all_probs  = np.array(all_probs)
# all_labels = np.array(all_labels)
# preds      = (all_probs >= 0.5).astype(int)
# tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()

# s1_metrics = {
#     "patch_auc"        : float(roc_auc_score(all_labels, all_probs)),
#     "patch_f1"         : float(f1_score(all_labels, preds, zero_division=0)),
#     "patch_sensitivity": float(recall_score(all_labels, preds, zero_division=0)),
#     "patch_specificity": float(tn / (tn + fp)),
# }
# print("Stage 1 Patch-Level Val Metrics")
# for k, v in s1_metrics.items():
#     print(f"  {k:25s}: {v:.4f}")

In [17]:
# feature_extractor = patch_model.backbone
# feature_extractor.eval()
# for p in feature_extractor.parameters():
#     p.requires_grad_(False)

# print("Extracting train features...")
# feats_tr  = extract_features(X_tr,  feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# print(f"  train : {feats_tr.shape}")

# print("Extracting val features...")
# feats_val = extract_features(X_val, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# print(f"  val   : {feats_val.shape}")

In [18]:
# import gc

# del X_tr, X_val
# gc.collect()
# torch.cuda.empty_cache()

# print(f"Running 5-fold CV for Stage 2 ({MODEL_NAME})...")
# feats_all = extract_features(X_train_all, feature_extractor, _mean_gpu, _std_gpu, DEVICE)

# del X_train_all
# gc.collect()

In [19]:
# cv_results, cv_summary = run_cv(
#     feats_all, y_train_all, bag_ids_all, fold_ids, feat_dim=FEAT_DIM,
#     attn_dim=ATTN_DIM, dropout=0.25, gated=True, tag=MODEL_NAME,
#     device=DEVICE
# )
# print(cv_summary)

# with open(OUT / f"{MODEL_NAME}_cv_results.json", "w") as f:
#     json.dump({
#         "model": MODEL_NAME, "feat_dim": FEAT_DIM,
#         "fold_results": cv_results,
#         "cv_mean": cv_summary.loc["mean"].to_dict(),
#         "cv_std": cv_summary.loc["std"].to_dict(),
#     }, f, indent=2)
# print("Saved:", f"{MODEL_NAME}_cv_results.json")

In [20]:
# from sklearn.metrics import ConfusionMatrixDisplay

# fig, axes = plt.subplots(1, 5, figsize=(20, 4))
# for i, r in enumerate(cv_results):
#     cm = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])
#     disp = ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"])
#     disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
#     axes[i].set_title(f"Fold {i}\nMCC={r['mcc']:.3f}")
# plt.suptitle(f"{MODEL_NAME} — per-fold confusion matrices (val, calibrated threshold)")
# plt.tight_layout()
# plt.savefig(OUT / f"{MODEL_NAME}_confusion_matrices.png", dpi=150)
# plt.show()